# 🕸️ Web Scraping in Python — Live Demo
### Data Science Using Python

**Site we'll scrape today:** [quotes.toscrape.com](https://quotes.toscrape.com/) — a website built *specifically* for people to practice scraping on. It's free, legal, and safe to hit as many times as you want.

By the end of this notebook, you will:
1. Understand what web scraping actually is
2. Fetch a real webpage using Python
3. Read HTML and pull out only the data you want
4. Turn that data into a clean table (DataFrame)
5. Save it as a CSV file you can open in Excel

> 💡 **How to use this in Colab:** Run each cell one by one using **Shift + Enter**. Don't skip ahead — each step builds on the one before it.

## 1. Why do we need web scraping? (5 min)

**Question for the class:** *"Suppose I want the quotes and authors of 100 famous quotes from a website. Would you manually copy-paste each one?"*

**The manual way:**

```
Website → Open quote → Copy text → Copy author → Paste into Excel → Repeat 100 times 😓
```

**The web scraping way:**

```
Website → Python → Find required data → Extract data → Store in CSV/Excel
```

> **Web scraping** is the process of automatically collecting data from websites using a program, instead of doing it by hand.

## 2. What does a webpage *really* look like? (5 min)

When you open a website, your browser shows you a nicely designed page. But underneath, what the browser actually receives is **HTML code**.

Try this right now (on any website, including this one): **Right-click → Inspect**. You'll see something like:

```html
<h1>Python Programming</h1>
<p>Learn Python easily</p>
<div class="price">₹500</div>
```

Here's the mental model to keep in your head for the rest of this class:

```
Website → HTML → Python reads HTML → Python finds required elements → Data extracted
```

Go ahead and open **https://quotes.toscrape.com/** in a new tab, right-click on any quote, and choose **Inspect**. Notice that each quote sits inside a `<div class="quote">` block. Keep that tab open — we'll come back to it.

## 3. The three Python tools we need

We won't overwhelm you with a dozen libraries. For this first demo, just three:

| Library | Job | Think of it as... |
|---|---|---|
| **requests** | Downloads the webpage | *"Go and bring me the webpage."* |
| **BeautifulSoup** (from `bs4`) | Reads the HTML and finds what we need | *"Look inside the webpage and find this for me."* |
| **pandas** | Organizes the extracted data into a table and saves it | *"Put the scraped data into a spreadsheet."* |

```
requests      → Get webpage
BeautifulSoup → Extract data
pandas        → Organize & save data
```

## 4. Setup

Google Colab already has `requests`, `beautifulsoup4`, and `pandas` pre-installed, so we just need to import them. (The `pip install` line below is only needed if you ever run this outside Colab.)

In [ ]:
# Uncomment the line below only if a library is missing (not needed on Colab)
# !pip install requests beautifulsoup4 pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd

print("All libraries imported successfully ✅")

## 5. Step 1 — Can Python get the webpage?

This is our first, simplest goal. Nothing fancy — just ask Python to fetch the page.

In [ ]:
url = "https://quotes.toscrape.com/"

response = requests.get(url)

print(response.status_code)

**What just happened?**

- `requests.get(url)` means *"Python, go to this URL and get the webpage."*
- `response.status_code` tells us how it went.
- **200** means success — the webpage was fetched correctly. 🎉

(Other codes you may see elsewhere: `404` = page not found, `403` = access denied, `500` = server error.)

## 6. Step 2 — Look at the raw HTML

Let's peek at what Python actually received. It will be a *huge* wall of HTML — don't panic, that's normal!

In [ ]:
# Full HTML is stored in response.text
# We'll just print the first 800 characters so it doesn't flood the screen
print(response.text[:800])

This is the real HTML sent by the website. We don't want *all* of it — we only want specific pieces: the quotes and their authors. That's exactly what BeautifulSoup helps us do.

## 7. Step 3 — Bring in BeautifulSoup

In [ ]:
soup = BeautifulSoup(response.text, "html.parser")

print(soup.title)
print(soup.title.text)

- `BeautifulSoup(response.text, "html.parser")` converts the messy HTML text into a searchable Python object called `soup`.
- `soup.title` finds the `<title>` tag of the page.
- `soup.title.text` gives us just the text inside it, without the HTML tags.

This is your **first successful extraction** from a real webpage!

## 8. A few HTML tags you should recognize

You don't need to become an HTML expert. Just remember this much:

| HTML tag | Meaning |
|---|---|
| `<h1>` | Main heading |
| `<p>` | Paragraph |
| `<a>` | Link |
| `<div>` | A container/box that groups other elements |
| `<span>` | A small inline container, often used for a short bit of text |
| `<small>` | Smaller / fine-print text |

And in BeautifulSoup:

- `soup.find("h1")` → finds the **first** `<h1>` element
- `soup.find_all("div")` → finds **all** `<div>` elements as a list

That's genuinely enough HTML knowledge to scrape most simple websites.

## 9. Step 4 — Extract just the quote text

If you inspected the page earlier, you'd have noticed every quote is wrapped like this:

```html
<div class="quote">
    <span class="text">"The world as we have created it..."</span>
    <small class="author">Albert Einstein</small>
    ...
</div>
```

Let's grab just the quote text first, using `find_all()`.

In [ ]:
quotes = soup.find_all("span", class_="text")

for q in quotes:
    print(q.text)

`soup.find_all("span", class_="text")` means: *"Find all `<span>` elements whose class is `text`."*

You should see 10 quotes printed — that's exactly how many are shown on the first page of the website.

## 10. Step 5 — Extract quote **and** author together (structured data)

Right now we only have quote text. Let's grab the *container* for each quote (`<div class="quote">`) so we can pull multiple related pieces of information — text **and** author — from the same block.

In [ ]:
url = "https://quotes.toscrape.com/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

quote_blocks = soup.find_all("div", class_="quote")

for block in quote_blocks:
    text = block.find("span", class_="text").text
    author = block.find("small", class_="author").text
    print(text, "-", author)

Notice what just happened: we've converted an **unstructured webpage** into **structured information** — pairs of (quote, author). This is exactly the mindset shift that turns 'reading a webpage' into 'collecting a dataset.'

## 11. Step 6 — This is where Data Science enters 📊

Let's collect everything into a Python list, and then hand it to **pandas** to build a proper table.

In [ ]:
data = []

for block in quote_blocks:
    text = block.find("span", class_="text").text
    author = block.find("small", class_="author").text
    data.append([text, author])

df = pd.DataFrame(data, columns=["Quote", "Author"])
df

```
Web Page → HTML → BeautifulSoup → Python list → Pandas DataFrame → Data Science
```

That's the **"aha!" moment** — a website has become a clean, structured table, ready for any kind of analysis.

## 12. Step 7 — Save the data as a CSV file

In [ ]:
df.to_csv("quotes.csv", index=False)
print("Data saved successfully! ✅")

In Colab, click the **📁 folder icon** on the left sidebar to see `quotes.csv` appear. You can double-click to preview it, or download it to open in Excel.

If you want it to download straight to your laptop, run the cell below.

In [ ]:
from google.colab import files
files.download("quotes.csv")

## 13. 🎯 Bonus: scraping *tags* too, and going beyond page 1

Every quote on the site also has **tags** (like `love`, `life`, `humor`), and the site has **multiple pages**. Let's push a little further — this is a great "stretch" example once you are comfortable with the basics.

In [ ]:
url = "https://quotes.toscrape.com/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

quote_blocks = soup.find_all("div", class_="quote")

data = []
for block in quote_blocks:
    text = block.find("span", class_="text").text
    author = block.find("small", class_="author").text
    tags = [tag.text for tag in block.find_all("a", class_="tag")]
    data.append([text, author, ", ".join(tags)])

df = pd.DataFrame(data, columns=["Quote", "Author", "Tags"])
df

### Scraping every page (pagination)

The website has a **"Next →"** link at the bottom of each page, going all the way up to page 10. We can loop through all of them automatically.

In [ ]:
all_data = []
url = "https://quotes.toscrape.com/page/1/"

while url:
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    quote_blocks = soup.find_all("div", class_="quote")
    for block in quote_blocks:
        text = block.find("span", class_="text").text
        author = block.find("small", class_="author").text
        all_data.append([text, author])

    # Find the "Next" button, if it exists
    next_btn = soup.find("li", class_="next")
    if next_btn:
        next_page = next_btn.find("a")["href"]
        url = "https://quotes.toscrape.com/" + next_page
    else:
        url = None   # no more pages, stop the loop

all_quotes_df = pd.DataFrame(all_data, columns=["Quote", "Author"])
print(f"Total quotes collected: {len(all_quotes_df)}")
all_quotes_df.head(10)

We just scraped **all 10 pages automatically** — around 100 quotes — using the exact same ideas from earlier, just wrapped in a loop.

## 14. 📝 Try it yourself (student exercise)

Now it's your turn. Using everything above, write code to:

1. Scrape **only the quotes tagged `"love"`** — hint: visit `https://quotes.toscrape.com/tag/love/` in your browser first and inspect it. The HTML structure is identical to the homepage!
2. Save the result as `love_quotes.csv`

Use the empty cell below.

In [ ]:
# 🧑‍💻 Your code here
url = "https://quotes.toscrape.com/tag/love/"

# Step 1: Get the page


# Step 2: Parse with BeautifulSoup


# Step 3: Find all quote blocks and extract text + author


# Step 4: Build a DataFrame and save to CSV


## 15. ✅ Your complete beginner scraping program

Here is the entire demo, start to finish, as one clean script.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Step 1: Website URL
url = "https://quotes.toscrape.com/"

# Step 2: Get the webpage
response = requests.get(url)

# Step 3: Convert HTML into a BeautifulSoup object
soup = BeautifulSoup(response.text, "html.parser")

# Step 4: Find all quote blocks
quote_blocks = soup.find_all("div", class_="quote")

# Step 5: Extract data
data = []
for block in quote_blocks:
    text = block.find("span", class_="text").text
    author = block.find("small", class_="author").text
    data.append([text, author])

# Step 6: Create a DataFrame
df = pd.DataFrame(data, columns=["Quote", "Author"])

# Step 7: Display data
print(df)

# Step 8: Save data
df.to_csv("quotes.csv", index=False)
print("\nData saved successfully!")

## 📌 Recap

| Step | Tool | What happened |
|---|---|---|
| Fetch the page | `requests` | Website → HTML text |
| Parse the HTML | `BeautifulSoup` | HTML text → searchable object |
| Find the data | `.find()` / `.find_all()` | Locate specific tags/classes |
| Organize it | `pandas` | List → DataFrame (table) |
| Save it | `.to_csv()` | DataFrame → CSV file |

### ⚖️ A quick word on ethics
- Always check a site's `robots.txt` and terms of service before scraping it for real projects.
- `quotes.toscrape.com` and `books.toscrape.com` are built *for practice* — they're always fair game.
- Never hammer a real website with rapid repeated requests; add delays (`time.sleep()`) for larger jobs.